In [7]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [13]:
MODEL = "gpt-4.1-mini"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

In [14]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

In [15]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature = 0, model_name=MODEL)

In [16]:
retriever.invoke("Who is Avery?")

[Document(id='da387fe0-ef70-4f89-b035-1c135a2b9d05', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [17]:
llm.invoke("Who is Avery?")

AIMessage(content='Could you please provide more context about which Avery you are referring to? There are many people and characters named Avery, so additional details would help me give you the most accurate information.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 11, 'total_tokens': 47, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_0e7b5db390', 'id': 'chatcmpl-EebeBGYxpRmNF8qWopDqDIiARwpr1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--09199d2d-138b-430b-8d49-6cebb1a49bdb-0', usage_metadata={'input_tokens': 11, 'output_tokens': 36, 'total_tokens': 47, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_detail

In [18]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [19]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [21]:
answer_question("Who is Avery?", [])

"Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She co-founded the company in 2015 and has played a key role in guiding it to become a leading provider in the Insurance Technology sector. Avery is recognized for her innovative leadership, risk management expertise, and commitment to diversity and community engagement. She has a strong background in product management from her previous role at Innovate Insurance Solutions and has been instrumental in driving Insurellm's growth and market presence. Avery is based in San Francisco, California, and currently earns a salary of $225,000."

In [22]:
gr.ChatInterface(answer_question).launch()

/Users/jaymomaya/Dev/Learning/RAG LLM QLoRA/learning-llm/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
